# CMES 92459 R1: main experiments (Tier 1)
Run each cell in order (Shift+Enter). The whole notebook takes about 2–3 minutes.
The last cell checks every key number against the manuscript and prints PASS/FAIL.

## 1. Get the code
The cell clones the public repository. To use your own Google Drive copy instead, set `USE_DRIVE = True`.

In [ ]:
USE_DRIVE = False
REPO_URL = 'https://github.com/USERNAME/pso-bezier-bspline-ode'   # replace USERNAME
import os, shutil
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SRC = '/content/drive/MyDrive/Claude Project/Numerical Method ODE & PDE/PSO - Verhulst Growth Solver/paper/CMES_R1/reproduce'
    shutil.copytree(SRC, '/content/reproduce', dirs_exist_ok=True); os.chdir('/content/reproduce')
else:
    !git clone -q {REPO_URL} /content/repo
    os.chdir('/content/repo/experiments')
print(sorted(f for f in os.listdir() if f.endswith('.py')))

In [ ]:
import numpy, scipy, matplotlib, platform
print('Python', platform.python_version(), '| NumPy', numpy.__version__, '| SciPy', scipy.__version__, '| Matplotlib', matplotlib.__version__)

## 2. PSO diagnosis and fix (Table 8)
Ignore the summary table printed at the end of `item1_pso_fix.py`: it uses a purely relative success test, which shows 0/30 for B-spline equilibrium. Read the output of `item1_analyse.py` instead.

In [ ]:
!python item1_pso_fix.py

In [ ]:
!python item1_analyse.py

## 3. Verification of the reference optimum (Table 7)

In [ ]:
!python itemA_optimum.py

## 4. Figures 1–7 and dense-grid errors (Table 6)

In [ ]:
!python regen_figs.py

## 5. Minimax objective (Table 16)

In [ ]:
!python itemC_minimax.py

## 6. Automatic check against the manuscript

In [ ]:
import json
ok = True
def chk(name, got, exp, tol=None):
    global ok
    good = (abs(got-exp) <= tol*abs(exp)) if tol else (got == exp)
    ok &= good
    print(f"{'PASS' if good else 'FAIL'}  {name:45s} got {got:<12.4g} expected {exp:.4g}")
S = {(r['basis'], r['regime'], r['variant']): r for r in json.load(open('item1_summary.json'))['rows']}
for reg, a in [('growth', 13), ('equilibrium', 18), ('decline', 18)]:
    chk(f'B-spline {reg}: original PSO successes', S['bspline', reg, 'A_original']['succ'], a)
    chk(f'B-spline {reg}: corrected PSO successes', S['bspline', reg, 'D_both']['succ'], 30)
for (b, reg), e in {('bezier','growth'): 6.47e-5, ('bezier','decline'): 7.62e-3, ('bspline','growth'): 2.90e-7, ('bspline','decline'): 2.03e-4}.items():
    chk(f'{b} {reg}: median max error', S[b, reg, 'D_both']['median'], e, tol=0.01)
A = json.load(open('itemA_optimum.json'))
for k, v in A.items():
    chk(f'{k}: LM starts at optimum (fraction)', v['lm_frac_at_opt'], 1.0)
    if 'grid_local_minima' in v: chk(f'{k}: grid local minima', v['grid_local_minima'], 1)
C = json.load(open('itemC_minimax.json'))
for k, e in {'bezier/growth': 30, 'bezier/decline': 30, 'bspline/growth': 24, 'bspline/decline': 29}.items():
    chk(f'minimax {k}: PSO successes', C[k]['methods']['PSO']['succ'], e)
print('\nALL CHECKS PASSED' if ok else '\nSOME CHECKS FAILED - send the output above to Claude')